In [53]:
import pandas as pd

In [ ]:
df = pd.read_csv('D:/Lomba/DATATHON DICODING/inflasiwatch/data/final/df_merged_bandung.xlsx')

In [55]:
import pandas as pd
import numpy as np

# sort by date
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

In [56]:
target_cols = [
    "ihk_umum_mtm",
    "inflasi_general_mtm",
    "inflasi_core_mtm",
    "inflasi_administered_price_mtm",
    "inflasi_volatile_good_mtm"
]

leading_cols = [
    "gt_harga_beras",
    "gt_harga_cabai",
    "gt_harga_telur",
    "gt_harga_daging_ayam",
    "gt_harga_minyak_goreng",
    "gt_harga_bawang",
    "gt_tiket_pesawat",
    "gt_tiket_kereta",
    "gt_harga_bensin",
    "kurs_usd_idr",
    "bi_rate_percent",
    "harga_bbm_pertamax",
    "oceanic_nino_index"
]

In [57]:
for col in target_cols + leading_cols:
    for lag in [1, 2, 3]:
        df[f"{col}_lag{lag}"] = df[col].shift(lag)

In [58]:
for col in target_cols:
    for window in [3, 6]:
        df[f"{col}_roll_mean_{window}"] = (
            df[col].rolling(window).mean()
        )
        
        df[f"{col}_roll_std_{window}"] = (
            df[col].rolling(window).std()
        )

In [59]:
gt_cols = [col for col in df.columns if col.startswith("gt_")]

for col in gt_cols:
    df[f"{col}_diff1"] = df[col].diff(1)
    df[f"{col}_pct_change"] = df[col].pct_change()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22576\1805648046.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_pct_change"] = df[col].pct_change()
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22576\1805648046.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_diff1"] = df[col].diff(1)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22576\1805648046.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Conside

In [60]:
df["month"] = df["date"].dt.month

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22576\1264498655.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["month"] = df["date"].dt.month
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22576\1264498655.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22576\1264498655.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Co

In [ ]:
ramadan_months = [
    "2020-04", "2020-05",
    "2021-04", "2021-05",
    "2022-04",
    "2023-03", "2023-04",
    "2024-03", "2024-04",
    "2025-03",
    "2026-02", "2026-03"
]

lebaran_months = [
    "2020-05",
    "2021-05",
    "2022-05",
    "2023-04",
    "2024-04",
    "2025-03",
    "2026-03"
]

df["year_month"] = df["date"].dt.strftime("%Y-%m")

df["is_ramadan"] = df["year_month"].isin(ramadan_months).astype(int)
df["is_lebaran"] = df["year_month"].isin(lebaran_months).astype(int)

In [ ]:
df["pre_lebaran_1m"] = df["is_lebaran"].shift(-1).fillna(0).astype(int)

In [ ]:
df["is_dry_season"] = df["month"].isin([6, 7, 8, 9]).astype(int)
df["is_rainy_season"] = df["month"].isin([11, 12, 1, 2]).astype(int)

In [ ]:
df["is_el_nino"] = (df["oceanic_nino_index"] >= 0.5).astype(int)
df["is_la_nina"] = (df["oceanic_nino_index"] <= -0.5).astype(int)

In [ ]:
df["bbm_price_hike_event"] = (
    df["harga_bbm_pertamax"].diff() > 0
).astype(int)

In [ ]:
df["bi_rate_hike_event"] = (
    df["bi_rate_percent"].diff() > 0
).astype(int)

In [ ]:
df.to_excel("D:/Lomba/DATATHON DICODING/inflasiwatch/data/processed/feature_engineer/feature_engineer_bandung.xlsx", index=False)